In [ ]:
import re
import numpy as np

KW = ["todo", "fixme", "hack", "workaround", "temp", "quick"]
_re_word = re.compile(r"\b\w+\b", re.UNICODE)
_re_upper = re.compile(r"[A-Z]")
_re_digit = re.compile(r"\d")
_re_punct = re.compile(r"[^\w\s]")
_re_snake = re.compile(r"\b[a-z]+_[a-z0-9_]+\b")
_re_camel = re.compile(r"\b[a-z]+[A-Z][A-Za-z0-9]*\b")

def cheap_features(text: str) -> dict:
    t = "" if text is None else str(text)
    tl = t.lower()

    n_chars = len(t)
    words = _re_word.findall(t)
    n_words = len(words) if words else 0

    n_upper = len(_re_upper.findall(t))
    n_digit = len(_re_digit.findall(t))
    n_punct = len(_re_punct.findall(t))

    pct_upper = n_upper / n_chars if n_chars else 0.0
    pct_digit = n_digit / n_chars if n_chars else 0.0
    pct_punct = n_punct / n_chars if n_chars else 0.0

    feats = {
        "n_chars": n_chars,
        "n_words": n_words,
        "pct_upper": pct_upper,
        "pct_digit": pct_digit,
        "pct_punct": pct_punct,
        "has_should": int(" should " in f" {tl} "),
        "has_later": int(" later " in f" {tl} "),
        "has_for_now": int(" for now " in tl),
        "has_snake_case": int(_re_snake.search(t) is not None),
        "has_camel_case": int(_re_camel.search(t) is not None),
        "has_path": int(("/" in t) or ("\\" in t)),
        "has_equals": int("=" in t),
        "has_brackets": int(("(" in t) or (")" in t) or ("{" in t) or ("}" in t) or ("[" in t) or ("]" in t)),
    }

    # keyword counts + flags
    for k in KW:
        feats[f"kw_{k}_cnt"] = tl.count(k)
        feats[f"kw_{k}_has"] = int(k in tl)

    return feats

In [ ]:
import pandas as pd

TEXT_COL = "orig_text"   # o "text"
df_feats = df.copy()
cheap = df_feats[TEXT_COL].apply(cheap_features).apply(pd.Series)
df_feats = pd.concat([df_feats, cheap], axis=1)

In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split

TARGET_COL = "logit_td"
LABEL_COL = "label"

num_cols = [c for c in df_feats.columns if c.startswith("n_") or c.startswith("pct_") or c.startswith("kw_") or c.startswith("has_")]
feature_cols = [TEXT_COL] + num_cols

train_df, valid_df = train_test_split(
    df_feats,
    test_size=0.2,
    random_state=42,
    stratify=df_feats[LABEL_COL] if LABEL_COL in df_feats.columns else None
)

train_pool = Pool(train_df[feature_cols], label=train_df[TARGET_COL], text_features=[TEXT_COL])
valid_pool = Pool(valid_df[feature_cols], label=valid_df[TARGET_COL], text_features=[TEXT_COL])

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=6000,
    learning_rate=0.05,
    depth=8,
    early_stopping_rounds=200,
    random_seed=42,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)